# 🧠 Advanced Local Agent: GUI + Memory + Observability (LangGraph)

This notebook upgrades your agent with:

✅ Gradio UI (GUI)
✅ Long-term memory (file-based)
✅ Observability (logging / tracing)
✅ LangGraph fundamentals

All fully **offline**.


In [ ]:
%pip install openai langgraph gradio

## 🧩 Architecture Overview

Components:

- **LLM** → Ollama (Qwen)
- **LangGraph** → agent loop
- **Memory** → JSON file store
- **Observability** → logs + debug prints
- **UI** → Gradio

---


In [ ]:
from openai import OpenAI
from langgraph.graph import StateGraph, END
import json, os
import gradio as gr

# -----------------------------------
# LLM CLIENT
# -----------------------------------
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# -----------------------------------
# MEMORY (simple file-based)
# -----------------------------------
MEMORY_FILE = "memory.json"

def load_memory():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    return []

def save_memory(msgs):
    with open(MEMORY_FILE, "w") as f:
        json.dump(msgs, f)

# -----------------------------------
# STATE
# -----------------------------------
class State(dict):
    messages: list

# -----------------------------------
# TOOLS
# -----------------------------------
def add_numbers(a: int, b: int):
    return {"result": a + b}

TOOLS = {"add_numbers": add_numbers}

# -----------------------------------
# LLM NODE
# -----------------------------------
def llm_node(state: State):
    print("[LLM] Thinking...")

    resp = client.chat.completions.create(
        model="qwen3:1.7b",
        messages=state["messages"],
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "add_numbers",
                    "description": "Add two integers",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "a": {"type": "integer"},
                            "b": {"type": "integer"}
                        },
                        "required": ["a", "b"]
                    }
                }
            }
        ],
        tool_choice="auto"
    )

    msg = resp.choices[0].message
    return {"messages": state["messages"] + [msg]}

# -----------------------------------
# TOOL NODE
# -----------------------------------
def tool_node(state: State):
    msg = state["messages"][-1]

    # No tool call → stop graph cleanly
    if not msg.tool_calls:
        return {"__end__": True}

    print("[TOOL] Executing tool...")

    call = msg.tool_calls[0]
    args = json.loads(call.function.arguments)

    result = add_numbers(**args)

    tool_msg = {
        "role": "tool",
        "content": json.dumps(result),
        "tool_call_id": call.id
    }

    return {"messages": state["messages"] + [tool_msg]}

# -----------------------------------
# GRAPH
# -----------------------------------
graph = StateGraph(State)
graph.add_node("llm", llm_node)
graph.add_node("tool", tool_node)

graph.set_entry_point("llm")
graph.add_edge("llm", "tool")
graph.add_edge("tool", "llm")

agent = graph.compile()

# -----------------------------------
# RUN FUNCTION
# -----------------------------------
def chat(user_input):
    memory = load_memory()

    memory.append({"role": "user", "content": user_input})

    state = {"messages": memory}
    out = agent.invoke(state)

    final = out["messages"][-1]["content"]

    memory.append({"role": "assistant", "content": final})
    save_memory(memory)

    return final

# -----------------------------------
# GRADIO UI
# -----------------------------------
ui = gr.Interface(
    fn=chat,
    inputs="text",
    outputs="text",
    title="Local AI Agent (LangGraph + Memory)",
)

ui.launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[TOOL] Executing tool...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[TOOL] Executing tool...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking...
[LLM] Thinking

In [ ]:
from openai import OpenAI
from langgraph.graph import StateGraph, END
import json, os
import gradio as gr

# -----------------------------------
# LLM CLIENT
# -----------------------------------
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# -----------------------------------
# MEMORY (simple file-based)
# -----------------------------------
MEMORY_FILE = "memory.json"

def load_memory():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    return []

def save_memory(msgs):
    with open(MEMORY_FILE, "w") as f:
        json.dump(msgs, f)

# -----------------------------------
# STATE
# -----------------------------------
class State(dict):
    messages: list

# -----------------------------------
# TOOLS
# -----------------------------------
def add_numbers(a: int, b: int):
    return {"result": a + b}

TOOLS = {"add_numbers": add_numbers}

# -----------------------------------
# LLM NODE
# -----------------------------------
def llm_node(state: State):
    print("[LLM] Thinking...")

    resp = client.chat.completions.create(
        model="qwen3:1.7b",
        messages=state["messages"],
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "add_numbers",
                    "description": "Add two integers",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "a": {"type": "integer"},
                            "b": {"type": "integer"}
                        },
                        "required": ["a", "b"]
                    }
                }
            }
        ],
        tool_choice="auto"
    )

    msg = resp.choices[0].message
    return {"messages": state["messages"] + [msg]}

# -----------------------------------
# TOOL NODE
# -----------------------------------
def tool_node(state: State):
    msg = state["messages"][-1]

    # No tool call → stop graph cleanly
    if not msg.tool_calls:
        return {"__end__": True}

    print("[TOOL] Executing tool...")

    call = msg.tool_calls[0]
    args = json.loads(call.function.arguments)

    result = add_numbers(**args)

    tool_msg = {
        "role": "tool",
        "content": json.dumps(result),
        "tool_call_id": call.id
    }

    return {"messages": state["messages"] + [tool_msg]}

# -----------------------------------
# GRAPH
# -----------------------------------
graph = StateGraph(State)
graph.add_node("llm", llm_node)
graph.add_node("tool", tool_node)

graph.set_entry_point("llm")

# FIX: no infinite loop
graph.add_edge("llm", "tool")
graph.add_edge("tool", END)

agent = graph.compile()

# -----------------------------------
# RUN FUNCTION
# -----------------------------------
def chat(user_input):
    memory = load_memory()

    memory.append({"role": "user", "content": user_input})

    state = {"messages": memory}
    out = agent.invoke(state)

    final = out["messages"][-1]["content"]

    memory.append({"role": "assistant", "content": final})
    save_memory(memory)

    return final

# -----------------------------------
# GRADIO UI
# -----------------------------------
ui = gr.Interface(
    fn=chat,
    inputs="text",
    outputs="text",
    title="Local AI Agent (LangGraph + Memory)",
)

ui.launch()


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


[LLM] Thinking...
[LLM] Thinking...
[TOOL] Executing tool...
[LLM] Thinking...


## 🔍 Observability (Offline)

We added:

- Console logs (`print`)
- Tool execution traces
- State tracking via memory file

### You can extend with:
- LangSmith (offline alt logging)
- Custom logging middleware
- SQLite tracking
